# Capstone: Refresh / Content Opportunity Scoring

This notebook contains the end-to-end pipeline for the final capstone project. It loads the starter dataset, engineers features, trains a Random Forest model, compares it to a baseline rule, and outputs a ranked queue.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
data_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)

# Filter out absolute zero impression pages
df = df[df['impressions_90d'] > 0].copy()
print(f"Loaded and filtered data: {df.shape[0]} rows")

Loaded and filtered data: 30000 rows


In [2]:
# 2. Define Features and Label
features = ['content_age_days', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count']
label = 'is_declining'

# Target proxy: is the trend direction down?
df[label] = (df['trend_direction'].str.lower() == 'down').astype(int)

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df[label]

print(f"Class balance (Declining vs Not):\n{y.value_counts(normalize=True).round(3)}")

Class balance (Declining vs Not):
is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


In [3]:
# 3. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 24000, Test size: 6000


In [4]:
# 4. Baseline Rule (Human Heuristic)
# Intuition: old page + highly visible = needs update
stale = (X_test['days_since_last_update'] >= 180).astype(int)
visible = (X_test['impressions_90d'] >= 500).astype(int)
baseline_scores = stale * visible * X_test['impressions_90d']

In [5]:
# 5. Train ML Model (Random Forest)
model = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

In [6]:
# 6. Evaluation (Precision@K)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("--- Results on Holdout Test Set ---")
for k in [20, 50, 100]:
    base_p = precision_at_k(baseline_scores, y_test.values, k)
    mod_p = precision_at_k(model_scores, y_test.values, k)
    print(f"Precision@{k:<3} | Baseline Rule: {base_p:.3f} | ML Model: {mod_p:.3f}")

--- Results on Holdout Test Set ---
Precision@20  | Baseline Rule: 0.550 | ML Model: 0.850
Precision@50  | Baseline Rule: 0.480 | ML Model: 0.880
Precision@100 | Baseline Rule: 0.510 | ML Model: 0.900


In [7]:
# 7. Generate Ranked Output Queue
df['model_decay_probability'] = model.predict_proba(X)[:, 1]

# Filter to actionable pages and sort
actionable_queue = df[df['impressions_90d'] > 100].sort_values('model_decay_probability', ascending=False)

print("\n--- Top 5 Recommended Pages for Editorial Review ---")
display(actionable_queue[['content_id', 'model_decay_probability', 'impressions_90d', 'days_since_last_update', 'trend_direction']].head(5))


--- Top 5 Recommended Pages for Editorial Review ---


,content_id,model_decay_probability,impressions_90d,days_since_last_update,trend_direction
12200,content_e88c269d36d5,0.698048,2099,104,down
6842,content_91fefd1726b3,0.696961,2748,104,down
21247,content_6dbee08d40f8,0.696435,176,104,down
15117,content_bb6ebb5ec8c8,0.696435,2621,104,down
20608,content_d829395fda44,0.696435,296,104,down
